# Protein to Network and Response Calculation Example

This notebook demonstrates how to:
1. Load a protein structure from the PDB using BioPython.
2. Create an elastic network model using the `elastory` library.
3. Define a set of beads (based on residue numbers) for perturbation.
4. Calculate the network's response to a concentric pull on these beads.
5. Visualize the initial and final structures.

In [ ]:
import numpy as np

from Bio.PDB.PDBList import PDBList
from Bio.PDB.PDBParser import PDBParser

from elastory.network.network import Network

## 1. Load PDB Structure (1n5u) using BioPython

In [ ]:
pdb_id = "1n5u"
pdbl = PDBList()
# Download PDB file
pdb_file_path = pdbl.retrieve_pdb_file(
    pdb_code=pdb_id, pdir="./pdb/", file_format="pdb", overwrite=True
)

# Parse the PDB file
parser = PDBParser(QUIET=True)
structure = parser.get_structure(pdb_id, pdb_file_path)

# Extract C-alpha coordinates and residue numbers
initial_coords_list = []
protein_ca_resnums = []

assert structure is not None, f"Failed to parse PDB file: {pdb_file_path}"
model = structure[0]  # Assuming first model
chain_id_to_select = "A"
selected_chain_ca_atoms = []

if chain_id_to_select in model:
    chain = model[chain_id_to_select]
    for residue in chain:
        # Standard residues have hetfield ' ', non-standard (HETATM) have other values
        if residue.id[0] == " " and "CA" in residue:  # Check for standard residue and CA atom
            selected_chain_ca_atoms.append(residue["CA"])
            protein_ca_resnums.append(residue.id[1])  # Residue sequence number
else:
    print(f"Warning: Chain '{chain_id_to_select}' not found in {pdb_id}.")

if not selected_chain_ca_atoms:
    print(f"Warning: No C-alpha atoms found in chain '{chain_id_to_select}'. Trying all chains.")
    protein_ca_resnums = []  # Reset for collecting from all chains
    for chain in model:
        for residue in chain:
            if residue.id[0] == " " and "CA" in residue:
                selected_chain_ca_atoms.append(residue["CA"])
                protein_ca_resnums.append(residue.id[1])

if not selected_chain_ca_atoms:
    raise ValueError(f"Could not extract C-alpha atoms from {pdb_id} using BioPython.")

initial_coords = np.array([atom.get_coord() for atom in selected_chain_ca_atoms])
centered_coords = initial_coords - np.mean(initial_coords, axis=0)

print(f"Loaded {pdb_id}, selected {len(centered_coords)} C-alpha atoms using BioPython.")

## 2. Create Elastic Network Model

In [ ]:
cutoff_distance = 9.0  # Angstroms
network = Network(bead_positions=centered_coords, cutoff_length=cutoff_distance, identifier=pdb_id)

print(
    f"Network created with {network.N} beads and {np.sum(np.diag(network.laplacian))} springs (cutoff = {cutoff_distance} Å)."
)

## 3. Define Beads for Concentric Pull (Heme Binding Site)

In [ ]:
heme_residue_numbers = [
    114,
    118,
    123,
    138,
    139,
    142,
    146,
    149,
    154,
    157,
    158,
    161,
    165,
    189,
    190,
    193,
]
# protein_ca_resnums is defined in the PDB loading cell

heme_site_indices = []
for i, resnum in enumerate(protein_ca_resnums):
    if resnum in heme_residue_numbers:
        heme_site_indices.append(i)

print(
    f"Selected {len(heme_site_indices)} beads for concentric pull (heme site indices): {heme_site_indices[:5]}..."
)

## 4. Calculate Network Response (Concentric Pull)

In [ ]:
num_steps = 100
network.max_pull = 0.8  # Pulls beads until their RoG is 20% of original for that group

network.source = heme_site_indices

trajectory = network.calculate_response_concentric(
    preload_steps=num_steps, noise_strength=0, progress_bar=True
)

## 5. Visualize Initial and Final Structures

In [ ]:
## Choose the general theme for plots.
plot_theme = "dark"
# plot_theme = 'light'

if plot_theme == "dark":
    plotly_theme = "plotly_dark"
    jtplot_theme = "onedork"
if plot_theme == "light":
    plotly_theme = "plotly"
    jtplot_theme = "grade3"

In [ ]:
from elastory.plot.plotly.utils import response_video, update_layout
import plotly.graph_objects as go

fig = go.Figure()
fig = response_video(fig, network, trajectory, speed=10)
fig = update_layout(fig, hide_axes=True, show_legend=False, theme=plotly_theme)
fig.show()